In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Paradoxia/opendata-iisys-hui", 
    repo_type="dataset", local_dir="./opendata-iisys-hui", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 149 files: 100%|██████████| 149/149 [00:48<00:00,  3.10it/s]


'/home/ubuntu/opendata-iisys-hui'

In [3]:
files = glob('opendata-iisys-hui/*/*.parquet')
len(files)

149

In [5]:
df = pd.read_parquet(files[0])
df

,speaker,book,audio,text
0,Ragnar,schwle_tage,{'bytes': b'RIFF\xb2\x91\x0e\x00WAVEfmt \x10\x...,"Dann kam mein Vater, in seinen weißen Staubman..."
1,Ragnar,schwle_tage,{'bytes': b'RIFF\x04\xd8\x0b\x00WAVEfmt \x10\x...,Auf der Fahrt unterhielt er mich liebenswürdig...
2,Ragnar,schwle_tage,{'bytes': b'RIFFj\x14\x1e\x00WAVEfmt \x10\x00\...,"Er freute sich über die gute Partie, die Ellit..."
3,Ragnar,schwle_tage,{'bytes': b'RIFF|\x1c\x13\x00WAVEfmt \x10\x00\...,In Warnow saß die Tante in großer Toilette unt...
4,Ragnar,schwle_tage,{'bytes': b'RIFF:\xde\n\x00WAVEfmt \x10\x00\x0...,Die Mädchen trugen weiße Kleider und Rosen im ...
...,...,...,...,...
642,Alexandra_Bogensperger,jane_eyre_die_waise_von_lowood,{'bytes': b'RIFF:\x87\x18\x00WAVEfmt \x10\x00\...,Ich sehnte mich nach einem Laib Brot. Durch so...
643,Alexandra_Bogensperger,jane_eyre_die_waise_von_lowood,{'bytes': b'RIFFh\xc3\x06\x00WAVEfmt \x10\x00\...,"Ich fühlte, daß es entehrend sei, an der Dorfs..."
644,Alexandra_Bogensperger,jane_eyre_die_waise_von_lowood,{'bytes': b'RIFF:M\x07\x00WAVEfmt \x10\x00\x00...,"Besaß ich denn nichts, was ich jenen Leuten zu..."
645,Alexandra_Bogensperger,jane_eyre_die_waise_von_lowood,{'bytes': b'RIFFL\x83\x08\x00WAVEfmt \x10\x00\...,Ich dachte nach. Um den Hals hatte ich ein kle...


In [6]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker'].iloc[i]}"
            })
        
    return data

In [7]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [02:08<00:00, 128.07s/it]


In [8]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 7/7 [11:18<00:00, 96.99s/it] 


In [9]:
len(data)

96331

In [10]:
data[0]

{'audio_filename': 'opendata-iisys-hui_audio/opendata-iisys-hui-data-train-00009-of-00149_0.mp3',
 'text': 'Dann kam mein Vater, in seinen weißen Staubmantel gehüllt, das Gesicht ein wenig gerötet vom Waschen: Du schimpfst wohl schon, sagte er lustig.',
 'speaker': 'opendata-iisys-hui_audio_Ragnar'}

In [11]:
with open('opendata-iisys-hui.json', 'w') as fopen:
    json.dump(data, fopen)

In [12]:
audio_files = [d['audio_filename'] for d in data]

with open('opendata-iisys-hui-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [13]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'opendata-iisys-hui_audio/opendata-iisys-hui-data-train-00009-of-00149_0.mp3',
 'text': 'Dann kam mein Vater, in seinen weißen Staubmantel gehüllt, das Gesicht ein wenig gerötet vom Waschen: Du schimpfst wohl schon, sagte er lustig.',
 'speaker': 'opendata-iisys-hui_audio_Ragnar'}

In [14]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'opendata-iisys-hui')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 19.08ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 9.08MB / 9.12MB, 45.3MB/s  
Processing Files (1 / 1): 100%|██████████| 9.12MB / 9.12MB, 22.8MB/s  
Processing Files (1 / 1): 100%|██████████| 9.12MB / 9.12MB, 9.11MB/s  
New Data Upload: 100%|██████████| 9.12MB / 9.12MB, 9.11MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.45s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/9b629f1209a8320ff44125363933bfd025e7c34a', commit_message='Upload dataset', commit_description='', oid='9b629f1209a8320ff44125363933bfd025e7c34a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [18]:
# !zip -rq opendata-iisys-hui_audio.zip opendata-iisys-hui_audio

In [19]:
# !hf upload malaysia-ai/Multilingual-TTS opendata-iisys-hui_audio.zip --repo-type=dataset

In [22]:
# !zip -rq opendata-iisys-hui_audio_neucodec.zip opendata-iisys-hui_audio_neucodec

In [23]:
# !hf upload malaysia-ai/Multilingual-TTS opendata-iisys-hui_audio_neucodec.zip --repo-type=dataset